# ragk8s — RAGAS judge `gemma4:12b-it-q8_0` (Ollama) on Kaggle

Scores the generation runs (faithfulness, answer_relevancy, context_precision,
context_recall) with a judge from another model family than the generators.
Every score is cached: a session can stop at any time and the next one resumes.

**Before running**
1. *Accelerator*: **GPU T4 x2**. *Internet*: **on**.
2. *Add Input*: the dataset made from the archive created locally with
   `python -m ragk8s kaggle export --preset main --runs qwen3.8-27b qwen2.5-7b-instruct --with-judge-cache --name judge-input`.
3. Set `MODE` below: `"pilot"` (about 20 minutes, measures the speed) or `"full"`.
   For `"full"`, use *Save Version → Save & Run All* so it runs in the background.

**After running**: download `ragk8s-judge-output.zip` from *Output*, then locally
`python -m ragk8s kaggle import <path>/ragk8s-judge-output.zip`.

In [ ]:
MODE = "pilot"          # "pilot" or "full"
MAX_MINUTES = 20 if MODE == "pilot" else 600   # Kaggle sessions are limited to 12 h
CONCURRENCY = 4         # parallel judge requests (OLLAMA_NUM_PARALLEL)
JUDGE_MODEL = "gemma4:12b-it-q8_0"

In [ ]:
import json, os, pathlib, subprocess, sys, time, urllib.request

manifests = list(pathlib.Path("/kaggle/input").rglob("manifest.json"))
assert len(manifests) == 1, f"attach exactly one ragk8s export dataset, found {manifests}"
DATA = manifests[0].parent
MANIFEST = json.loads(manifests[0].read_text())
os.environ["RAGK8S_GIT_COMMIT"] = MANIFEST["commit"]
print(json.dumps(MANIFEST, indent=2))
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=True)

Install `ragk8s` with the pinned judge dependencies, at the exact commit of the export.

In [ ]:
pkg = f"ragk8s[judge] @ git+{MANIFEST['repo']}@{MANIFEST['commit']}"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import importlib.metadata as md
print({p: md.version(p) for p in ("ragk8s", "ragas", "langchain-community", "openai", "sentence-transformers")})

Install and start Ollama (4 parallel requests, 8k context), then pull the judge.

In [ ]:
subprocess.run("apt-get -qq update && apt-get -qq install -y zstd > /dev/null", shell=True, check=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
env = {**os.environ, "OLLAMA_KEEP_ALIVE": "-1", "OLLAMA_NUM_PARALLEL": str(CONCURRENCY), "OLLAMA_CONTEXT_LENGTH": "8192"}
server = subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"),
                          stderr=subprocess.STDOUT, env=env)
for _ in range(60):
    try:
        print(urllib.request.urlopen("http://127.0.0.1:11434/api/version").read().decode())
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start, see /kaggle/working/ollama.log")
subprocess.run(["ollama", "pull", JUDGE_MODEL], check=True)
subprocess.run(["ollama", "list"], check=True)

Smoke test: one JSON request with thinking disabled (the answer must be plain JSON, quickly).

In [ ]:
from openai import OpenAI
client = OpenAI(base_url="http://127.0.0.1:11434/v1", api_key="ollama")
t = time.time()
r = client.chat.completions.create(
    model=JUDGE_MODEL, temperature=0, reasoning_effort="none", response_format={"type": "json_object"},
    messages=[{"role": "user", "content": 'Return {"answer": <the capital of France>} as JSON.'}])
print(round(time.time() - t, 1), "s |", r.choices[0].message.content, "|", r.usage)
subprocess.run(["ollama", "ps"], check=True)

Copy the runs (and the judge cache of previous sessions), then score.

In [ ]:
import shutil
WORK = pathlib.Path("/kaggle/working/rag")
for sub in ("results/runs", "results/retrieval", "eval", ".cache/judge"):
    if (DATA / sub).exists():
        shutil.copytree(DATA / sub, WORK / sub, dirs_exist_ok=True)
os.chdir(WORK)
runs = sorted(str(p.relative_to(WORK)) for p in WORK.glob("results/runs/*/run_*.jsonl"))
print(len(runs), "runs:", runs)

def judge(models, preset, extra=()):
    cmd = [sys.executable, "-m", "ragk8s", "judge", "--models", *models, "--preset", preset,
           "--max-minutes", str(MAX_MINUTES), "--concurrency", str(CONCURRENCY), *extra]
    subprocess.run(cmd, check=True)

if MODE == "pilot":
    judge(["qwen3.8-27b"], "best", ["--limit", "10"])
else:
    judge(["qwen3.8-27b"], "main")
    judge(["qwen2.5-7b-instruct"], "best")

Measured speed and remaining work (from the score metadata).

In [ ]:
metas = [json.loads(p.read_text()) for p in sorted(WORK.glob("results/judge/*/scores_*.meta.json"))]
for m in metas:
    print(m["run"], {k: v["ok"] for k, v in m["status"].items()}, "errors:", {k: v["error"] for k, v in m["status"].items()})
print("judge:", metas[0]["judge"], "| last session:", metas[0]["last_session"], "| s per new score:", metas[0]["seconds_per_new_score"])

In [ ]:
import zipfile
out = pathlib.Path("/kaggle/working/ragk8s-judge-output.zip")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ("results/judge", ".cache/judge"):
        for f in (WORK / folder).rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
print(out, round(out.stat().st_size / 1024), "KiB")